In [ ]:
import anndata
import numpy as np
import pandas as pd
# Set pandas to display all columns
import os
import tifffile
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
import scanpy as sc
pd.set_option("display.max_columns", None)

In [ ]:
BASE_PATH = Path('/Volumes/BiomedML/Projects/QuPath_projects/Polaris/mask_skin1/')
CELL_PATH = BASE_PATH.joinpath('Skin_1_cell_measurement.txt')
IMAGE_PATH = BASE_PATH.joinpath('stitched_image.ome.tif-.tiff')

In [ ]:
CELL_PATH

In [ ]:
cell_measure = pd.read_csv(CELL_PATH, sep='\t')

In [ ]:
cell_measure.head()

In [ ]:
cell_coords = cell_measure[['Centroid X µm', 'Centroid Y µm']]


In [ ]:
# fig = plt.figure(figsize=(10,10))
# ax = sns.scatterplot(x=cell_coords['Centroid X µm'], y=cell_coords['Centroid Y µm'], c=cell_measure['Cell: PanCK (Opal 690) mean'],cmap='gray',s=3)
# ax.invert_yaxis()
# plt.gray()
# ax.imshow(mask, cmap='gray', alpha=0.75)

fig = plt.figure(figsize=(10,10))
ax = sns.scatterplot(x=cell_coords['Centroid X µm'], y=cell_coords['Centroid Y µm'], 
                     hue=np.log(cell_measure['Cell: PanCK (Opal 690) mean']+1),palette='RdBu',cmap='viridis' ,s=1)
ax.invert_yaxis()
norm = plt.Normalize(np.log(cell_measure['Cell: PanCK (Opal 690) mean']+1).min(), np.log(cell_measure['Cell: PanCK (Opal 690) mean']+1).max())
sm = plt.cm.ScalarMappable(cmap="RdBu", norm=norm)
sm.set_array([])
ax.figure.colorbar(sm)
plt.gray()

In [ ]:
fig = plt.figure(figsize=(10,10))
ax = sns.scatterplot(x=cell_coords['Centroid X µm'], y=cell_coords['Centroid Y µm'], 
                     hue=np.log(cell_measure['Cell: CD8 (Opal 480) mean']+1),palette='RdBu',cmap='viridis' ,s=1)
ax.invert_yaxis()
norm = plt.Normalize(np.log(cell_measure['Cell: CD8 (Opal 480) mean']+1).min(), np.log(cell_measure['Cell: CD8 (Opal 480) mean']+1).max())
sm = plt.cm.ScalarMappable(cmap="RdBu", norm=norm)
sm.set_array([])
ax.figure.colorbar(sm)
plt.gray()

In [ ]:
# fig = plt.figure(figsize=(10,10))
# ax = sns.scatterplot(x=cell_coords['Centroid X µm'], y=cell_coords['Centroid Y µm'], c=cell_measure['Cell: PanCK (Opal 690) mean'],cmap='gray',s=3)
# ax.invert_yaxis()
# plt.gray()
# ax.imshow(mask, cmap='gray', alpha=0.75)

fig = plt.figure(figsize=(10,10))
ax = sns.scatterplot(x=cell_coords['Centroid X µm'], y=cell_coords['Centroid Y µm'], 
                     hue=np.log(cell_measure['Cell: PD-1 (Opal 620) mean']+1),palette='RdBu',cmap='viridis' ,s=1)
ax.invert_yaxis()
norm = plt.Normalize(np.log(cell_measure['Cell: PD-1 (Opal 620) mean']+1).min(), np.log(cell_measure['Cell: PD-1 (Opal 620) mean']+1).max())
sm = plt.cm.ScalarMappable(cmap="RdBu", norm=norm)
sm.set_array([])
ax.figure.colorbar(sm)
plt.gray()

In [ ]:
sns.scatterplot

In [ ]:
cell_measure.columns.values

In [ ]:
# Retain only MeanIntensity measurements
MeanIntensity_columns = [ col for col in cell_measure.columns.values if ('Cell: ' in col) and ('mean'in col)]
MeanIntensity_columns

In [ ]:
MeanIntensity_columns.remove('Cell: AF (Autofluorescence) mean')
target_names = MeanIntensity_columns.copy()

In [ ]:
markers =  [i.split(' ')[1] for i in target_names]
markers

In [ ]:
AreaShape_columns = [col for col in cell_measure.columns.values if 
                     'µm' in col or 'Area' in col or 'Perimeter' in col or 'Circularity' in col or
                    'caliper' in col or 'Eccentricity' in col or 'area ratio' in col ]
AreaShape_columns

In [ ]:
cells = cell_measure[AreaShape_columns + MeanIntensity_columns]
cells = cells.drop(['Cell: AF (Autofluorescence) mean'],axis=1)

In [ ]:
cells.describe()

In [ ]:

col_name_mapper = dict(zip(target_names, markers))
cells = cells.rename(columns=col_name_mapper)

In [ ]:
cells.dtypes

In [ ]:
cells.hist(column=markers, figsize=(12,12), bins=20)

In [ ]:
adata = anndata.AnnData(cells[markers])
adata.obs = cells[[col for col in cells.columns.values if col not in markers]]
adata.obs

In [ ]:
adata.var_names.shape

In [ ]:
adata.write(BASE_PATH.joinpath('cell_matrix_obj.h5ad'), compression='gzip')

In [ ]:
df = adata.to_df()
df.boxplot(figsize=(10, 10), rot=90)

In [ ]:
adata.X = df.to_numpy()

In [ ]:
sc.pl.highest_expr_genes(adata)

In [ ]:
sc.pp.log1p(adata)
sc.pl.highest_expr_genes(adata)

In [ ]:
adata_markers = adata[:, adata.var_names.drop(['DAPI'])]
sc.tl.pca(adata_markers, svd_solver='arpack')
sc.pl.pca(adata_markers)

In [ ]:
sc.pp.neighbors(adata_markers, n_neighbors=15, n_pcs=5)
sc.tl.umap(adata_markers)

In [ ]:
sc.pl.umap(adata_markers)

In [ ]:
sc.tl.leiden(adata_markers, resolution=0.07)

sc.pl.umap(adata_markers, color=['leiden'], size=8)


In [ ]:
all_genes_marker = adata_markers.var_names
sc.pl.dotplot(adata_markers, all_genes_marker, groupby='leiden') #,save='20201017_tophat_disk_dotplot.pdf')


In [ ]:
fig = plt.figure(figsize=(10,10))
ax = sns.scatterplot(x=adata_markers.obs['Centroid X µm'], y=adata_markers.obs['Centroid Y µm'], 
                     hue=adata_markers.['leiden'], s=10)
ax.invert_yaxis()

In [ ]:
#Cells that express both marker
# PD1-PD-L1, FoxP3-CD68

adata_marker.X